In [1]:
quiet_library <- function(...) { suppressPackageStartupMessages(library(...)) }
quiet_library(GEOquery)
quiet_library(hise)
quiet_library(purrr)
quiet_library(data.table)
quiet_library(dplyr)

Warning message:
“package ‘GEOquery’ was built under R version 4.4.2”
Warning message:
“package ‘Biobase’ was built under R version 4.4.2”
Warning message:
“package ‘BiocGenerics’ was built under R version 4.4.2”
Warning message:
“package ‘purrr’ was built under R version 4.4.2”
Warning message:
“package ‘data.table’ was built under R version 4.4.3”


In [2]:
quiet_library(reticulate)
use_python("/home/workspace/environment/minimal/bin/python")
anndata <- import("anndata")

Warning message:
“package ‘reticulate’ was built under R version 4.4.3”


In [3]:
if(!dir.exists("output")) {
    dir.create("output")
}

### Gene Metadata

BRI used UCSC GRC38.91 for alignment.

In [4]:
ensembl_url <- "https://ftp.ensembl.org/pub/release-91/gtf/homo_sapiens/Homo_sapiens.GRCh38.91.chr_patch_hapl_scaff.gtf.gz"
download.file(ensembl_url, "Homo_sapiens.GRCh38.91.chr_patch_hapl_scaff.gtf.gz")
system("gunzip Homo_sapiens.GRCh38.91.chr_patch_hapl_scaff.gtf.gz")

In [5]:
gtf <- fread(
    "Homo_sapiens.GRCh38.91.chr_patch_hapl_scaff.gtf", 
    sep = "\t", 
    skip = 5,
    header = FALSE
) %>%
  as.data.frame()

In [6]:
gtf_names <- c("seqname", "source", "feature", "start", "end", "score", "strand", "frame", "attributes")
names(gtf) <- gtf_names

In [7]:
nrow(gtf)

[1] 2847484

In [8]:
gene_meta <- gtf %>%
  mutate(id = sub("gene_id ([^;]+);.+","\\1", attributes),
         name = ifelse(
             grepl("gene_name", attributes),
             sub(".+gene_name ([^;]+);.+","\\1", attributes),
             id
         )) %>%
  dplyr::select(id, name) %>%
  base::unique()

In [9]:
gene_meta$name <- make.unique(gene_meta$name)

In [10]:
nrow(gene_meta)

[1] 63967

In [11]:
gene_meta <- gene_meta %>%
  mutate(name = gsub('"','',name),
         id = gsub('"','',id))

In [12]:
row.names(gene_meta) <- gene_meta$name

### Sample Metadata

In [13]:
meta_uuid <- 'af25e3e7-25c1-4476-afb4-926bd201db8f'
meta_file <- cacheFiles(list(meta_uuid))
sample_meta <- read.csv(meta_file)[,-1]

[1] "downloading fileID af25e3e7-25c1-4476-afb4-926bd201db8f"


In [14]:
head(sample_meta)

,cohort.cohortGuid,subject.subjectGuid,subject.biologicalSex,subject.cmv,subject.bmi,subject.race,subject.ethnicity,subject.birthYear,subject.ageAtFirstDraw,subject.covidVaxDose1.daysSinceFirstVisit,subject.covidVaxDose2.daysSinceFirstVisit,sample.sampleKitGuid,sample.visitName,sample.drawDate,sample.subjectAgeAtDraw,sample.daysSinceFirstVisit,specimen.specimenGuid,pipeline.fileGuid
,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>
1,BR1,BR1001,Female,Negative,23,Caucasian,Non-Hispanic origin,1987,32,NA,NA,KT00001,Flu Year 1 Day 0,2019-10,32,0,PB00001-01,fec489f9-9a74-4635-aa91-d2bf09d1faec
2,BR1,BR1002,Male,Negative,22,Caucasian,Non-Hispanic origin,1991,28,440,461,KT00002,Flu Year 1 Day 0,2019-10,28,0,PB00002-01,7c0c7979-eebd-4aba-b5b2-6e76b4643623
3,BR1,BR1003,Female,Negative,21,Caucasian,Non-Hispanic origin,1989,30,440,461,KT00003,Flu Year 1 Day 0,2019-10,30,0,PB00003-01,40efd03a-cb2f-4677-af42-a056cbfe5a17
4,BR1,BR1004,Male,Negative,22,Caucasian,Non-Hispanic origin,1989,30,543,563,KT00004,Flu Year 1 Day 0,2019-10,30,0,PB00004-01,68fbcd34-1d63-461d-8195-df5b8dc61b31
5,BR1,BR1005,Female,Negative,20,Caucasian,Non-Hispanic origin,1992,27,451,492,KT00006,Flu Year 1 Day 0,2019-10,27,0,PB00006-01,ea8d98e9-e99e-4dc6-9e78-9866e0deac68
6,BR1,BR1003,Female,Negative,21,Caucasian,Non-Hispanic origin,1989,30,440,461,KT00007,Flu Year 1 Day 7,2019-10,30,8,PB00007-01,1faf2b5f-66e4-4787-8a8b-487621fc4c08


Select relevant columns to retain

In [15]:
sample_meta <- sample_meta %>%
  mutate(sample.drawYear = sub("-.+", "", sample.drawDate)) %>%
  dplyr::select(-sample.drawDate, -specimen.specimenGuid, -pipeline.fileGuid)

Add ageGroup and arrange columns

In [16]:
sample_meta <- sample_meta %>%
  mutate(subject.ageGroup = ifelse(
      cohort.cohortGuid == "BR1",
      "Young Adult", "Older Adult")) %>%
  dplyr::select(starts_with("cohort"), starts_with("subject"), starts_with("sample"))

Prepare for combination with the GEO data by making a conversion column for timepoints

In [17]:
timepoint_conversion <- data.frame(
    geo.timepoint = c(
        "Y1 Flu Day 0", "Y1 Flu Day 7-9", "Y1 Flu Day 80-100",
        "Y2 Flu Day 0", "Y2 Flu Day 7-9", "Y2 Flu Day 80-100",
        "Non-Flu Day 0", "Non-Flu Day 7-9", "Non-Flu Day 80-100",
        "Standalone Baseline Visit", "Standalone Baseline Visit", "Standalone Baseline Visit"),
    sample.visitName = c(
        "Flu Year 1 Day 0", "Flu Year 1 Day 7", "Flu Year 1 Day 90",
        "Flu Year 2 Day 0", "Flu Year 2 Day 7", "Flu Year 2 Day 90",
        "Immune Variation Day 0", "Immune Variation Day 7", "Immune Variation Day 90",
        "Flu Year 1 Stand-Alone", "Flu Year 2 Stand-Alone", "Flu Year 3 Stand-Alone")
)

In [18]:
sample_meta <- sample_meta %>%
  left_join(timepoint_conversion)

Joining with `by = join_by(sample.visitName)`


In [19]:
nrow(sample_meta)

[1] 868

### Unperturbed whole blood samples

In [20]:
series_id <- "GSE279552"

Get sample metadata stored in GEO

In [21]:
sre <- suppressMessages(getGEO(series_id, GSEMatrix = FALSE))

In [22]:
sample_info <- GSMList(sre)

In [23]:
sample_characteristics <- map_dfr(
  sample_info,
  function(si) {
    raw_chars <- Meta(si)$characteristics_ch1
    char_names <- c("geo_accession", "source_name", "description", "title",
                    sub(":.+","", raw_chars))
    char_vals <- c(Meta(si)$geo_accession,
                   Meta(si)$source_name_ch1,
                   Meta(si)$description[length(Meta(si)$description)],
                   Meta(si)$title,
                   sub(".+: ","", raw_chars))
    names(char_vals) <- char_names
    as.data.frame(as.list(char_vals))
  }
)

Rename to prep for combination with sample_meta

In [24]:
sample_characteristics <- sample_characteristics %>%
  dplyr::select(description, geo_accession, title, donor, timepoint) %>%
  dplyr::rename(barcodes = description,
                subject.subjectGuid = donor,
                geo.accession = geo_accession,
                geo.title = title,
                geo.timepoint = timepoint)

In [25]:
head(sample_characteristics)

,barcodes,geo.accession,geo.title,subject.subjectGuid,geo.timepoint
,<chr>,<chr>,<chr>,<chr>,<chr>
1,lib80149,GSM8575086,BR2035 Y1 Flu Day 0,BR2035,Y1 Flu Day 0
2,lib80150,GSM8575087,BR2035 Y1 Flu Day 7-9,BR2035,Y1 Flu Day 7-9
3,lib80151,GSM8575088,BR2035 Y1 Flu Day 80-100,BR2035,Y1 Flu Day 80-100
4,lib80152,GSM8575089,BR2035 Non-Flu Day 0,BR2035,Non-Flu Day 0
5,lib80153,GSM8575090,BR2035 Non-Flu Day 7-9,BR2035,Non-Flu Day 7-9
6,lib80154,GSM8575091,BR2035 Non-Flu Day 80-100,BR2035,Non-Flu Day 80-100


In [26]:
nrow(sample_characteristics)

[1] 864

Combine and filter based on sample_meta

Subjects that aren't included in sample_meta (and cause NA values in sample.sampleKitGuid) were either not truly Healthy, or had limited sample collection:  
BR1020: No flu vaccine time points  
BR1034: Psoriasis  
BR1045: One sample doesn't have corresponding scRNA-seq  
BR2007: Abnormal  
BR2049: Abnormal  


In [27]:
final_samples <- sample_characteristics %>%
  left_join(sample_meta, by = c("subject.subjectGuid","geo.timepoint")) %>%
  filter(!is.na(sample.sampleKitGuid))

In [28]:
nrow(final_samples)

[1] 838

In [29]:
rownames(final_samples) <- final_samples$barcodes

In [30]:
unstim_meta_csv <- paste0("output/sound-life_whole-blood_metadata_", Sys.Date(),".csv")

write.csv(
    final_samples,
    unstim_meta_csv,
    row.names = FALSE,
    quote = FALSE
)

### Assemble count matrix

In [31]:
supp_file <- getGEOSuppFiles(series_id)

Using locally cached version of supplementary file(s) GSE279552 found here:
/home/workspace/sound-life-scrna-analysis/04-file-sets/GSE279552/GSE279552_P462_genecounts.csv.gz 



In [32]:
mat <- fread(rownames(supp_file)) %>%
  as.data.frame()

In [33]:
dim(mat)

[1] 58302   865

In [34]:
ensembl_ids <- mat$V1
mat <- as.matrix(mat[,-1])
rownames(mat) <- ensembl_ids

In [35]:
sum(ensembl_ids %in% gene_meta$id)

[1] 58302

In [36]:
nrow(mat)

[1] 58302

In [37]:
mat <- mat[,final_samples$barcodes]

In [38]:
rownames(mat) <- gene_meta$name[match(rownames(mat), gene_meta$id)]

In [39]:
str(mat)

 int [1:58302, 1:838] 1 0 91 39 18 2105 12 37 12 71 ...
 - attr(*, "dimnames")=List of 2
  ..$ : chr [1:58302] "TSPAN6" "TNMD" "DPM1" "SCYL3" ...
  ..$ : chr [1:838] "lib80149" "lib80150" "lib80151" "lib80152" ...


In [40]:
unstim_mat_csv <- paste0("output/sound-life_whole-blood_matrix_", Sys.Date(),".csv")

write.csv(
    mat,
    unstim_mat_csv,
    row.names = FALSE,
    quote = FALSE
)

### Assemble .h5ad with AnnData

In [41]:
var <- gene_meta[rownames(mat),]

In [42]:
unstim_adata <- anndata$AnnData(
    X = t(mat),
    obs = final_samples,
    var = var
)

In [43]:
unstim_h5ad <- paste0("output/sound-life_whole-blood_", Sys.Date(),".h5ad")
unstim_adata$write_h5ad(unstim_h5ad)

### Perturbation whole blood samples

In [44]:
series_id <- "GSE279480"

Get sample metadata stored in GEO

In [45]:
sre <- suppressMessages(getGEO(series_id, GSEMatrix = FALSE))

In [46]:
sample_info <- GSMList(sre)

In [47]:
sample_characteristics <- map_dfr(
  sample_info,
  function(si) {
    raw_chars <- Meta(si)$characteristics_ch1
    char_names <- c("geo_accession", "source_name", "description", "title",
                    sub(":.+","", raw_chars))
    char_vals <- c(Meta(si)$geo_accession,
                   Meta(si)$source_name_ch1,
                   Meta(si)$description[length(Meta(si)$description)],
                   Meta(si)$title,
                   sub(".+: ","", raw_chars))
    names(char_vals) <- char_names
    as.data.frame(as.list(char_vals))
  }
)

Rename to prep for combination with sample_meta

In [48]:
sample_characteristics <- sample_characteristics %>%
  select(description, geo_accession, title, donor, timepoint, stimulation) %>%
  rename(barcodes = description) %>%
  rename(subject.subjectGuid = donor) %>%
  rename(geo.accession = geo_accession,
         geo.title = title,
         geo.timepoint = timepoint,
         geo.stimulation = stimulation)

In [49]:
head(sample_characteristics)

,barcodes,geo.accession,geo.title,subject.subjectGuid,geo.timepoint,geo.stimulation
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,lib73151,GSM8572211,BR2031 LPS Y1 Flu Day 7-9,BR2031,Y1 Flu Day 7-9,LPS
2,lib73152,GSM8572212,BR2031 Null Y1 Flu Day 7-9,BR2031,Y1 Flu Day 7-9,Null
3,lib73153,GSM8572213,BR2031 Poly I:C Y1 Flu Day 7-9,BR2031,Y1 Flu Day 7-9,Poly I:C
4,lib73154,GSM8572214,BR2031 SEB Y1 Flu Day 7-9,BR2031,Y1 Flu Day 7-9,SEB
5,lib73163,GSM8572215,BR1003 Null Standalone Baseline Visit,BR1003,Standalone Baseline Visit,Null
6,lib73164,GSM8572216,BR1003 Poly I:C Standalone Baseline Visit,BR1003,Standalone Baseline Visit,Poly I:C


In [50]:
nrow(sample_characteristics)

[1] 1021

Combine and filter based on sample_meta

Subjects that aren't included in sample_meta (and cause NA values in sample.sampleKitGuid) were either not truly Healthy, or had limited sample collection:  
BR1034: Psoriasis  
BR2007: Abnormal  
BR2049: Abnormal  


In [51]:
final_samples <- sample_characteristics %>%
  left_join(sample_meta, by = c("subject.subjectGuid","geo.timepoint")) %>%
  filter(!is.na(sample.sampleKitGuid))

In [52]:
nrow(final_samples)

[1] 985

In [53]:
stim_meta_csv <- paste0("output/sound-life_whole-blood-stim_metadata_", Sys.Date(),".csv")

write.csv(
    final_samples,
    stim_meta_csv,
    row.names = FALSE,
    quote = FALSE
)

In [54]:
supp_file <- getGEOSuppFiles(series_id)

Using locally cached version of supplementary file(s) GSE279480 found here:
/home/workspace/sound-life-scrna-analysis/04-file-sets/GSE279480/GSE279480_P441_genecounts.csv.gz 



### Assemble count matrix

In [55]:
mat <- fread(rownames(supp_file)) %>%
  as.data.frame()

In [56]:
uniprot_ids <- mat$V1
mat <- as.matrix(mat[,-1])
rownames(mat) <- uniprot_ids

In [57]:
mat <- mat[,final_samples$barcodes]

In [58]:
rownames(mat) <- gene_meta$name[match(rownames(mat), gene_meta$id)]

In [59]:
str(mat)

 int [1:58302, 1:985] 7 0 182 34 1 2567 15 59 21 107 ...
 - attr(*, "dimnames")=List of 2
  ..$ : chr [1:58302] "TSPAN6" "TNMD" "DPM1" "SCYL3" ...
  ..$ : chr [1:985] "lib73151" "lib73152" "lib73153" "lib73154" ...


In [60]:
stim_mat_csv <- paste0("output/sound-life_whole-blood-stim_matrix_", Sys.Date(),".csv")

write.csv(
    mat,
    stim_mat_csv,
    row.names = FALSE,
    quote = FALSE
)

### Assemble .h5ad with AnnData

In [61]:
var <- gene_meta[rownames(mat),]

In [62]:
stim_adata <- anndata$AnnData(
    X = t(mat),
    obs = final_samples,
    var = var
)

In [63]:
stim_h5ad <- paste0("output/sound-life_whole-blood-stim_", Sys.Date(),".h5ad")
stim_adata$write_h5ad(stim_h5ad)

## Upload data to HISE

In [71]:
study_space_uuid <- "de025812-5e73-4b3c-9c3b-6d0eac412f2a"
title <- paste("Sound Life Whole Blood RNA-seq from GEO", Sys.Date())

In [72]:
search_id <- ids::proquint(n_words = 3)
search_id

[1] "zajar-vatop-midip"

In [73]:
in_list <- list(meta_uuid)

In [74]:
out_list <- list(
    unstim_meta_csv, unstim_mat_csv, unstim_h5ad,
    stim_meta_csv, stim_mat_csv, stim_h5ad
)

In [75]:
out_list

[[1]]
[1] "output/sound-life_whole-blood_metadata_2025-07-10.csv"

[[2]]
[1] "output/sound-life_whole-blood_matrix_2025-07-10.csv"

[[3]]
[1] "output/sound-life_whole-blood_2025-07-10.h5ad"

[[4]]
[1] "output/sound-life_whole-blood-stim_metadata_2025-07-10.csv"

[[5]]
[1] "output/sound-life_whole-blood-stim_matrix_2025-07-10.csv"

[[6]]
[1] "output/sound-life_whole-blood-stim_2025-07-10.h5ad"

In [76]:
uploadFiles(
    files = out_list,
    studySpaceId = study_space_uuid,
    title = title,
    inputFileIds = in_list,
    store = "project",
    destination = search_id
)

[1] "Retrying..."
[1] "/home/workspace/environment/minimal"
[1] "Cannot determine the current notebook."
[1] "1) /home/workspace/sound-life-scrna-analysis/04-file-sets/R_assemble_whole_blood_GEO_data.ipynb"
[1] "2) /home/workspace/sound-life-scrna-analysis/04-file-sets/R_assemble_pseudobulk_csv.ipynb"
[1] "3) /home/workspace/sound-life-scrna-analysis/04-file-sets/output/Untitled.ipynb"


Please select (1-3)  1


$Message
[1] "General Okay-ness"

$VisualizationId
[1] "00000000-0000-0000-0000-000000000000"

$AbstractionId
[1] "00000000-0000-0000-0000-000000000000"

$TraceId
[1] "9c4a2bbb-57e5-4d42-b73d-058289a73d17"

$ProcessId
[1] "baf6d8c8-21db-4658-a389-33177bb1e378"

$WorkflowId
[1] "43053b37-f250-49a8-8d90-3f0891bdb537"

$FileIds
$FileIds[[1]]
[1] "81cf71c4-fdba-4208-a968-2d4c01c62336"

$FileIds[[2]]
[1] "44e06d33-7c61-4c37-a8fa-b8e2f7f4d285"

$FileIds[[3]]
[1] "32efea09-fc6e-4e0f-aa88-48fd61515b26"

$FileIds[[4]]
[1] "af348f01-7e9f-4598-add9-5f37dd72ceff"

$FileIds[[5]]
[1] "0abffd4c-251c-4aa5-913c-b9c9272ffad5"

$FileIds[[6]]
[1] "9529c6a7-6258-43f7-9e4a-0992e7edca06"

In [77]:
sessionInfo()

R version 4.4.1 (2024-06-14)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS/LAPACK: /home/workspace/environment/minimal/lib/libopenblasp-r0.3.28.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8    LC_NUMERIC=C        LC_TIME=C          
 [4] LC_COLLATE=C        LC_MONETARY=C       LC_MESSAGES=C      
 [7] LC_PAPER=C          LC_NAME=C           LC_ADDRESS=C       
[10] LC_TELEPHONE=C      LC_MEASUREMENT=C    LC_IDENTIFICATION=C

time zone: America/Los_Angeles
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] reticulate_1.42.0   dplyr_1.1.4         data.table_1.17.6  
[4] purrr_1.0.4         hise_2.16.0         GEOquery_2.74.0    
[7] Biobase_2.66.0      BiocGenerics_0.52.0

loaded via a namespace (and not attached):
 [1] SummarizedExperiment_1.36.0 httr2_1.1.2                
 [3] lattice_0.22-6              tzdb_0.5.0 